# 02 · The reverse process and the loss

We can destroy data. Now: how do we *undo* it? This notebook derives what the
network should predict and why the training loss ends up being a plain
mean-squared error.

> ### 📝 How this notebook works
> Cells marked **`# TODO`** are yours to write. Each is followed by a
> **self-check** cell that verifies your implementation and prints ✅.
>
> **Stuck?** The answer key is `solutions/notebooks/`, and the reference
> implementation lives in the `nanodiffusion/` package. Peeking is allowed —
> but try first.


## 1. What we want, and why it's hard

To generate, we want to sample from the reverse of one step:
$q(x_{t-1}\mid x_t)$. By Bayes' rule,

$$q(x_{t-1}\mid x_t)=\frac{q(x_t\mid x_{t-1})\,q(x_{t-1})}{q(x_t)}$$

The trouble is $q(x_{t-1})$ — the distribution of all slightly-less-noisy images
in the universe. We don't have it. **So we learn an approximation instead.**

### The saving grace

There's a classical result: when each step is *small enough* (tiny $\beta_t$), the
reverse of a diffusion step is **also approximately Gaussian**. This is exactly
why we use many small steps rather than a few big ones. So we can model:

$$p_\theta(x_{t-1}\mid x_t)=\mathcal N\big(x_{t-1};\ \mu_\theta(x_t,t),\ \sigma_t^2 I\big)$$

The variance we'll just fix to a known constant. **All the network has to produce
is the mean $\mu_\theta$.**

## 2. The trick: condition on $x_0$

Here's the clever move. The reverse step is intractable — *but* if we pretend we
already know the clean sample $x_0$, it becomes an exact, closed-form Gaussian:

$$q(x_{t-1}\mid x_t, x_0)=\mathcal N\big(x_{t-1};\ \tilde\mu_t(x_t,x_0),\ \tilde\beta_t I\big)$$

with (you can get these by multiplying the two Gaussians and completing the square)

$$\tilde\mu_t(x_t,x_0)=\frac{\sqrt{\bar\alpha_{t-1}}\,\beta_t}{1-\bar\alpha_t}\,x_0+\frac{\sqrt{\alpha_t}\,(1-\bar\alpha_{t-1})}{1-\bar\alpha_t}\,x_t,\qquad \tilde\beta_t=\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t}\,\beta_t$$

Read $\tilde\mu_t$ as a **weighted average of "where we came from" ($x_0$) and
"where we are" ($x_t$)**. During training we *do* know $x_0$ — it's the data — so
this gives us an exact target to aim at.

### Why conditioning on $x_0$ actually fixes it

The reverse step $q(x_{t-1}\mid x_t)$ was intractable because, hidden inside it, is
the question *"which clean images could $x_t$ have come from?"* — which needs the
whole data distribution:

$$q(x_{t-1}\mid x_t)=\int \underbrace{q(x_{t-1}\mid x_t, x_0)}_{\text{easy}}\;\underbrace{q(x_0\mid x_t)}_{\text{hard: needs the data}}\,dx_0$$

The trick isolates the **easy** factor. Apply Bayes to $q(x_{t-1}\mid x_t, x_0)$ and
every piece becomes a forward-process Gaussian we *already know*:

$$q(x_{t-1}\mid x_t, x_0)=\frac{\overbrace{q(x_t\mid x_{t-1})}^{\text{one noising step}}\;\overbrace{q(x_{t-1}\mid x_0)}^{\text{nice property at }t-1}}{\underbrace{q(x_t\mid x_0)}_{\text{nice property at }t}}$$

(The first term drops the $x_0$ because the forward chain is **Markov**: given
$x_{t-1}$, the next dab of noise doesn't care where we started.) A product and
quotient of known Gaussians is Gaussian — complete the square and you get the
closed form above. The hard marginalization over $x_0$ hasn't disappeared; the
**network** will learn it implicitly, by training across many $x_0$ samples.

### The picture: a bridge between two anchors

Why does knowing $x_0$ help so much? The forward process is a random walk that
started at $x_0$. Asking *"where was the walk one step ago?"* from $x_t$ **alone**
is hopeless — it could have drifted in from anywhere. But pin down **both** ends —
where it started ($x_0$) and where it is now ($x_t$) — and the previous step is
essentially a point on the path *between* those two anchors, plus a little noise.

That is exactly why $\tilde\mu_t$ comes out as a **weighted blend of $x_0$ and
$x_t$**, and why its variance $\tilde\beta_t$ is *smaller* than $\beta_t$: two
fixed endpoints leave less room for uncertainty than one. (Formally, this is a
**Brownian bridge**.)

We're allowed to use $x_0$ because at **training** time we have it — it's the data
sample. The network then learns to reproduce this exact target from $x_t$ and $t$
alone, which is all it will have at **generation** time.

## 3. From $x_0$ to $\varepsilon$: why predict the noise ⭐

During *generation* we won't know $x_0$. But rearrange the nice property from
notebook 01:

$$x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\varepsilon \quad\Longrightarrow\quad x_0=\frac{x_t-\sqrt{1-\bar\alpha_t}\,\varepsilon}{\sqrt{\bar\alpha_t}}$$

We *do* know $x_t$ (we're holding it). So **knowing $\varepsilon$ is the same as
knowing $x_0$** — they're related by an invertible formula. Substituting this into
$\tilde\mu_t$ and simplifying, all the $x_0$ terms collapse into a strikingly
compact expression:

$$\boxed{\;\tilde\mu_t=\frac{1}{\sqrt{\alpha_t}}\Big(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\,\varepsilon\Big)\;}$$

Everything on the right is known **except $\varepsilon$**. So the entire generative
problem reduces to a single question:

> **Given a noisy $x_t$ and its timestep $t$, what noise was added?**

That is a plain supervised regression problem — and we can generate infinite
labelled training data for it, because *we* choose the noise.

### Why $\varepsilon$ rather than $x_0$ or $\mu$?

All three are mathematically equivalent (invertible transforms of each other), but
$\varepsilon$ is the best-behaved **target** for a neural net: it is
$\mathcal N(0,I)$ — zero mean, unit variance — at *every* noise level. Predicting
$x_0$ at high $t$, by contrast, means predicting something nearly impossible from
nearly pure noise, and the target scale varies wildly with $t$. Empirically,
$\varepsilon$-prediction trains much more stably.

## 4. The loss: from a scary ELBO to `mse_loss`

Formally we train by maximizing a variational lower bound (ELBO) on the
likelihood. It decomposes into a sum over timesteps of KL divergences:

$$L=\underbrace{\mathbb E_q\Big[\sum_{t>1} D_{\mathrm{KL}}\big(q(x_{t-1}|x_t,x_0)\ \|\ p_\theta(x_{t-1}|x_t)\big)\Big]}_{\text{match the true reverse step}}+\ \dots$$

That looks intimidating, but both distributions are **Gaussians with the same
fixed variance**, and the KL between two such Gaussians is just the squared
distance between their means:

$$D_{\mathrm{KL}}=\frac{1}{2\sigma_t^2}\big\|\tilde\mu_t-\mu_\theta\big\|^2$$

Substituting the boxed $\tilde\mu$ formula, the $x_t$ terms cancel and only the
noise difference survives, leaving a weighted MSE on $\varepsilon$:

$$L_t=\frac{\beta_t^2}{2\sigma_t^2\alpha_t(1-\bar\alpha_t)}\big\|\varepsilon-\varepsilon_\theta(x_t,t)\big\|^2$$

The DDPM authors then found something delightful: **throwing the ugly weight away
works better.** That gives the loss we actually use:

$$\boxed{\;L_{\text{simple}}=\mathbb E_{x_0,\,t,\,\varepsilon}\Big[\big\|\varepsilon-\varepsilon_\theta(x_t,t)\big\|^2\Big]\;}$$

(Intuitively, dropping the weight de-emphasises the very small $t$ terms and makes
the model spend more effort on the harder, noisier steps that matter more for
perceptual quality.)

**Why a random $t$ each step?** The true loss is a *sum over all $t$*. Computing
it fully every iteration would cost $T$ forward passes. Instead we sample one $t$
uniformly — an unbiased Monte-Carlo estimate of the sum. Over many steps it
averages out to the same thing, for $1/T$ the cost.

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from nanodiffusion.utils import pick_device, set_seed
from nanodiffusion.data import toy2d
from nanodiffusion.schedules import NoiseSchedule
from nanodiffusion.forward import add_noise          # the nice property, from nb 01
# reference, used ONLY by the self-check cells:
from nanodiffusion.models import SinusoidalTimeEmbedding as ReferenceTimeEmbedding

set_seed(0)
device = pick_device()
print("device:", device)
data = toy2d("swiss_roll", 8000).to(device)
schedule = NoiseSchedule.make("cosine", 200).to(device)

## 5. Telling the network *when* it is

One network handles **all** noise levels, so it must know which $t$ it's looking
at — removing a whisper of noise ($t=5$) is a very different job from
hallucinating structure out of static ($t=190$).

Feeding the raw integer `t = 137` works poorly: neural nets handle large,
unnormalized integers badly, and a single number gives the network no easy way to
be sensitive at multiple scales. Instead we use the **sinusoidal embedding** from
transformers — evaluate $t$ against many sine/cosine waves of geometrically spaced
frequencies:

$$\text{emb}(t)=\big[\sin(\omega_1 t),\dots,\sin(\omega_{d/2}t),\ \cos(\omega_1 t),\dots,\cos(\omega_{d/2}t)\big]$$

with $\omega_i=10000^{-i/(d/2-1)}$. Think of it as a **bank of clocks** ticking at
different speeds: the fast ones distinguish $t=137$ from $t=138$, the slow ones
distinguish "early" from "late". Together they give a smooth, information-rich
code where nearby timesteps get nearby embeddings — so the network generalizes
across noise levels rather than memorizing each one.

## TODO 1 — sinusoidal time embedding

For an even embedding dim $d$, with `half = d // 2`:

1. `freqs[i] = exp(-log(10000) * i / (half - 1))` for `i` in `0..half-1`
2. `args = t[:, None] * freqs[None, :]` → shape `(B, half)`
3. return `concat([sin(args), cos(args)], dim=-1)` → shape `(B, d)`

In [ ]:
class MyTimeEmbedding(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        assert dim % 2 == 0, "dim must be even"
        self.dim = dim

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        '''(B,) timesteps -> (B, dim) embedding.'''
        half = self.dim // 2
        # TODO: build `freqs` (use torch.arange(half, device=t.device)),
        #       then args = t.float()[:, None] * freqs[None, :],
        #       then return torch.cat([sin, cos], dim=-1)
        raise NotImplementedError

In [ ]:
# ---- self-check 1 ----
emb = MyTimeEmbedding(64).to(device)
t_probe = torch.arange(0, 200, device=device)
out = emb(t_probe)
ref = ReferenceTimeEmbedding(64).to(device)(t_probe)
assert out.shape == (200, 64), f"shape {tuple(out.shape)} != (200, 64)"
assert torch.allclose(out, ref, atol=1e-5), "doesn't match the reference"

plt.figure(figsize=(7, 2.5))
plt.imshow(out.detach().cpu().T, aspect="auto", cmap="RdBu")
plt.xlabel("timestep t"); plt.ylabel("embedding dimension")
plt.title("Your time embedding (fast clocks on top, slow below)")
plt.colorbar(); plt.show()
print("✅ TODO 1 correct")

## TODO 2 — the denoiser $\varepsilon_\theta(x_t, t)$

For 2D points a small MLP is plenty (the U-Net comes in Part 2, for images). The
architecture is given; implement `forward`:

1. embed the timestep: `temb = self.time_mlp(t)` → `(B, time_embed_dim)`
2. concatenate onto `x` along the last dim → `(B, 2 + time_embed_dim)`
3. run through `self.net` → `(B, 2)`, the predicted noise

In [ ]:
class MyDenoiser(nn.Module):
    def __init__(self, data_dim=2, hidden=128, depth=4, time_embed_dim=64):
        super().__init__()
        self.time_mlp = nn.Sequential(
            MyTimeEmbedding(time_embed_dim),
            nn.Linear(time_embed_dim, time_embed_dim),
            nn.SiLU(),
        )
        layers = [nn.Linear(data_dim + time_embed_dim, hidden), nn.SiLU()]
        for _ in range(depth - 1):
            layers += [nn.Linear(hidden, hidden), nn.SiLU()]
        layers += [nn.Linear(hidden, data_dim)]
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        '''Predict the noise eps in x at timesteps t.  x:(B,2) t:(B,) -> (B,2)'''
        # TODO: implement (3 lines: embed t, concat, run net)
        raise NotImplementedError

In [ ]:
# ---- self-check 2 ----
model = MyDenoiser().to(device)
xb, tb = data[:16], torch.randint(0, 200, (16,), device=device)
out = model(xb, tb)
assert out.shape == xb.shape, f"output {tuple(out.shape)} should match input {tuple(xb.shape)}"
assert out.requires_grad, "output should be part of the autograd graph"
print(f"✅ TODO 2 correct — {sum(p.numel() for p in model.parameters()):,} parameters")

## TODO 3 — the loss

The whole of §4, in four lines:

1. sample a random timestep per item — `torch.randint(0, T, (B,), device=...)`
2. noise the batch — `x_t, noise = add_noise(x0, t, schedule)`
3. predict — `pred = model(x_t, t)`
4. return `F.mse_loss(pred, noise)`

Note step 2 hands back the *exact* `noise` it used — that's our regression label.

In [ ]:
def my_ddpm_loss(model, x0: torch.Tensor, schedule: NoiseSchedule) -> torch.Tensor:
    '''DDPM epsilon-prediction MSE loss for one batch of clean data.'''
    # TODO: implement (4 lines, per the steps above)
    raise NotImplementedError

In [ ]:
# ---- self-check 3: can it overfit one batch? ----
# The classic debugging move: if the model can't memorize 128 points, something
# is wrong with the loss -- far better to find out now than after a long run.
set_seed(0)
probe = MyDenoiser().to(device)
opt = torch.optim.Adam(probe.parameters(), lr=2e-3)
batch = data[:128]

vals = []
for step in range(400):
    l = my_ddpm_loss(probe, batch, schedule)
    opt.zero_grad(); l.backward(); opt.step()
    vals.append(l.item())

# average over a window: the per-step loss is noisy because t is random each step
start, end = sum(vals[:20]) / 20, sum(vals[-20:]) / 20
print(f"loss: {start:.3f} -> {end:.3f}")
assert end < start, "loss did not decrease — check your loss function"
print("✅ TODO 3 correct — overfit check passed")

## 6. Train for real

Now the full loop on all 8000 points, with **your** loss.

**What loss should you expect?** Not zero! The target $\varepsilon$ is drawn
*independently* of $x_0$, so a large part of it is fundamentally unpredictable.
Predicting exactly 0 every time would give an MSE of 1.0 (the variance of
$\mathcal N(0,I)$). Getting down to ~0.4 means the model is genuinely extracting
information about the noise from $x_t$ — that residual is the irreducible part.
A steadily-falling loss that plateaus well under 1.0 is exactly what success looks
like here.

In [ ]:
set_seed(0)
model = MyDenoiser().to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)

losses = []
for step in range(2000):
    idx = torch.randint(0, data.shape[0], (512,), device=device)
    loss = my_ddpm_loss(model, data[idx], schedule)
    opt.zero_grad(); loss.backward(); opt.step()
    losses.append(loss.item())
    if step % 400 == 0:
        print(f"step {step:4d}   loss {loss.item():.4f}")

plt.figure(figsize=(6, 3))
plt.plot(losses, alpha=0.35)
plt.plot(torch.tensor(losses).unfold(0, 50, 1).mean(1), color="C1", label="smoothed")
plt.axhline(1.0, ls="--", c="gray", lw=1, label="predicting zero")
plt.xlabel("step"); plt.ylabel("MSE"); plt.legend(); plt.title("Training loss"); plt.show()

tail = sum(losses[-100:]) / 100      # smoothed, since single-step loss is noisy
print(f"mean loss over last 100 steps: {tail:.4f}")
assert tail < 0.6, "expected the loss to settle well below 0.6"
print("✅ trained")

In [ ]:
import os
os.makedirs("../checkpoints", exist_ok=True)
torch.save(model.state_dict(), "../checkpoints/my_toy_mlp.pt")
print("saved ../checkpoints/my_toy_mlp.pt — your very own trained diffusion model")

## Recap

- The reverse step is intractable, but **approximately Gaussian** for small
  $\beta_t$ — so we only need to learn its **mean**.
- Conditioning on $x_0$ gives an exact Gaussian posterior with a closed-form mean
  $\tilde\mu_t$.
- Rewriting $x_0$ in terms of $\varepsilon$ collapses that mean into a formula whose
  **only unknown is the noise** — so predicting $\varepsilon$ solves everything.
- The ELBO reduces to a **weighted MSE on $\varepsilon$**, and dropping the weights
  works better: $L_{\text{simple}}=\|\varepsilon-\varepsilon_\theta(x_t,t)\|^2$.

Next: plug $\varepsilon_\theta$ into the boxed $\tilde\mu$ formula and actually
generate.